# Solution C1 · The signal sits in the ninth component: PCR fails

**Type**: Break the model

## Objective

Build a dataset in which the target depends on a direction with **tiny variance** and verify that a regression on the first principal components has R² zero until you include that component (the ninth out of ten, with 0.2% of the variance). Then compare with the regression on the original features (which sees all the directions) and with PLS, and understand what "explained variance" really measures.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline


def generate():
    rng = np.random.default_rng(0)
    n, p = 600, 10
    sd = np.array([5, 4, 3, 2.5, 2, 1.5, 1, 0.6, 0.4, 0.3])
    Z = rng.normal(size=(n, p)) * sd
    Q, _ = np.linalg.qr(rng.normal(size=(p, p)))                    # rotation: the original variables are mixtures
    y = Z[:, 8] * 3 + rng.normal(0, 0.5, n)                          # only the ninth direction (sd 0.4) matters
    X = Z @ Q.T
    return X - X.mean(0), y


X, y = generate()
cv = KFold(5, shuffle=True, random_state=0)

## Commented solution

In [ ]:
r2_per_k = pd.Series({k: cross_val_score(make_pipeline(PCA(k), LinearRegression()), X, y, cv=cv, scoring="r2").mean() for k in range(1, 11)})
r2_per_k.round(3)

In [ ]:
pca = PCA().fit(X)
variance_ratio = pca.explained_variance_ratio_
scores = pca.transform(X)
corr_with_y = np.abs([np.corrcoef(scores[:, i], y)[0, 1] for i in range(10)])
useful_component = int(np.argmax(corr_with_y)) + 1
useful_component_variance = variance_ratio[useful_component - 1]
print("share of variance per component:", variance_ratio.round(3))
print("correlation with y per component:", corr_with_y.round(2))
print(f"the useful component is number {useful_component}, with {useful_component_variance:.2%} of the variance")

In [ ]:
full_r2 = cross_val_score(LinearRegression(), X, y, cv=cv, scoring="r2").mean()
min_useful_k = int(r2_per_k.index[(r2_per_k > 0.5).to_numpy().argmax()])
ridge_r2 = cross_val_score(RidgeCV(alphas=np.logspace(-4, 0, 9)), X, y, cv=cv, scoring="r2").mean()
pls1_r2 = cross_val_score(PLSRegression(n_components=1), X, y, cv=cv, scoring="r2").mean()
print(f"R2 with all the features: {full_r2:.3f} | first k with R2 > 0.5: {min_useful_k} | Ridge: {ridge_r2:.3f} | PLS with 1 component: {pls1_r2:.3f}")

## Check

In [ ]:
with verify():
    check(list(r2_per_k.index) == list(range(1, 11)) and r2_per_k.loc[1:8].max() < 0.05, "with 1-8 components the R2 is practically zero (below 0.05)", "PCA(k) + LinearRegression, cross-validated R2")
    check(r2_per_k[9] > 0.8 and r2_per_k[10] > 0.8, "with 9 components the R2 goes above 0.8", "include the ninth component")
    check(useful_component == 9 and useful_component_variance < 0.005 and corr_with_y[8] > 0.85 and (np.delete(corr_with_y, 8) < 0.2).all(), "the useful component is the ninth: 0.2% of the variance, correlation over 0.85 with y; the others below 0.2", "PCA().fit_transform(X) and np.corrcoef with y")
    check(min_useful_k == 9 and full_r2 > 0.8 and abs(full_r2 - r2_per_k[10]) < 1e-9, "minimum k 9; the regression on the 10 features (or on 10 components) is the same", "PCA(10) is just a rotation")
    check(ridge_r2 is not None and ridge_r2 > 0.8 and pls1_r2 is not None and pls1_r2 < 0.1, "Ridge on the original features sees the useful direction (R2 above 0.8); PLS with one component does not (below 0.1)", "RidgeCV with small alphas; PLSRegression(n_components=1)")
done()

## What you learned

- **PCA does not look at `y`.** With the first 8 components (99.6% of the variance) the R² is zero, with 9 it goes up to 0.85: all the useful information is in a direction that contains 0.2% of the variance. Keeping "95% of the variance" would have thrown away 100% of the signal.
- **"Explained variance" measures reconstruction fidelity, not usefulness for a target.** They are two different questions: the first concerns the columns (X), the second the link with `y`. They often coincide in real data, where the directions with the most variance are also the most informative, but there is no guarantee.
- **Ridge on the original features works (0.85)**: it discards no direction. **PLS with one component does not (0.02)**: it chooses the direction of maximum *covariance* with `y`, that is variance × correlation, and the high-variance directions with a small random correlation (0.05) beat the low-variance
  one with high correlation (0.91). No method that summarizes the features into a few directions is immune: the rule is not to throw away directions without checking that the downstream task does not get worse, and to choose the number of components with cross-validation.
- In real data this scenario shows up when the useful variable is a small difference between two large, correlated quantities (for example the difference between two nearly equal measurements): its variance is tiny and PCA eliminates it first.

## Common pitfall

Reducing the dimensions with PCA "keeping 95% of the variance" before a supervised model, without checking that the downstream performance does not get worse. The threshold on the variance says nothing about the target.

## Interview question

<details><summary>Why does PCA not guarantee that the chosen components are useful for predicting?</summary>

Because it is unsupervised: it orders the directions by variance of X and not by relationship with y. If the signal sits in a low-variance direction (a difference between two large, correlated variables, for example) it gets discarded first. That is why `k` must be chosen with
cross-validation on the final task, or you use supervised methods (PLS, LDA) or models that see the original features (Ridge, trees).

</details>